## Machine learning Project: Using Machine learning to distinguish Leptons from Hadrons using LHCb dataset

The first step is to load all the necceccary modules

In [35]:
#importing neccecary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn

Then we can load the dataset into pandas

In [36]:
#loading the dataset into pandas
print("Loading dataset...")
training_data = pd.read_csv("training.csv")

#checking if load was successful
print(f"Dataset successfully loaded: Total number of particle tracks: {len(training_data)}")
#print("First 5 rows of dataset:")
#print(training_data.head())

Loading dataset...
Dataset successfully loaded: Total number of particle tracks: 1200000


Since I'm going to be training the algorithm to decipher weather the track which is measured is that of a hadron or a lepton, I'm going to filter the dataset to remove all of the entries where 'Ghost' is seen.

In [37]:
#filters the dataset to remove rows where the label entry is ghost
data_filtered = training_data[training_data['Label'] != 'Ghost'].copy()


Now im going to introduce a new column which is a boolean variable of 1 if the particle is a lepton and 0 if its a hadron

In [38]:
binary_map = {
    'Electron': 1,  # 1 for leptons
    'Muon': 1,      
    'Pion': 0,  # 0 for hadrons
    'Kaon': 0,
    'Proton': 0
}

data_filtered['BinaryClass'] = data_filtered['Label'].map(binary_map)
ignore_cols = {'ID', 'Label', 'Class', 'BinaryClass'} # Ignoring all metadata colums so only numerical measurements remain
features = [col for col in data_filtered.columns if col not in ignore_cols]

#verification
print("\nBinary Class Distribution")
print(df_clean['BinaryClass'].value_counts())
print(f"Total input features selected: {len(features)}")


Binary Class Distribution
BinaryClass
0    600000
1    400000
Name: count, dtype: int64
Total input features selected: 49


Now im going to use StandardScaler to make sure that the features with massive numerical range (momentum) don't distort the error calculation

In [39]:
# First i extract the raw numpy feature matrices (X) and the target arrays (Y)
X = data_filtered[features].values
y = data_filtered['BinaryClass'].values

# Then i split the data in to training (80%) and validation (20%)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Applying the Scaler
scaler = StandardScaler()

# Fitting and transforming the training features 
# (using fit_transform so that the algorithm learns the statistical data from the training data)
X_train_scaled = scaler.fit_transform(X_train)

# Just transforming the validation data so that there is no data leakage
X_val_scaled = scaler.transform(X_val)

# verification
print(f"training features shape: {X_train_scaled.shape}")
print(f"training features shape: {X_val_scaled.shape}")

training features shape: (800000, 49)
training features shape: (200000, 49)


Now its time to start the actual machine learning!

In [40]:
def predict_particles(X, w, b):
    """
    This will calculate the linear predictions, y_hat on the dataset
    X = a 2D array of the detector features (rows=particles, columns=measurements)
    w = 1D array of the weights
    b = scalar bias term
    """
    y_hat = np.dot(X, w) + b
    return y_hat

Now i need to define a multi Layer perceptron using PyTorch. Im going to use 3 layers starting with an output of 64 features

In [41]:
class LeptonHadronNN(nn.Module):
    def __init__(self, input_dim=49): # because i have 49 elements in my dataset
        super(LeptonHadronNN, self).__init__()

        #Layer 1 going from 49 inputs to 64 neurons
        self.fc1 = nn.Linear(in_features=input_dim, out_features=64) # Linear class type because i started with linear equations y = wx + b
        self.relu = nn.ReLU()

        #Layer 2 going from the 64 neurons to 32
        self.fc2 = nn.Linear(in_features=64, out_features=32)

        #Layer 3 going from 32 to 1 (which is the output score required for binary classification of 1 or 0 (Lepton or Hadron))
        self.fc3 = nn.Linear(in_features=32, out_features=1)

    def forward(self, x):
        # This defines the forward flow so that i start with the 49 features and end up with 1
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.fc3(x)
        return x

model = LeptonHadronNN(input_dim=49)
print(model)

LeptonHadronNN(
  (fc1): Linear(in_features=49, out_features=64, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=64, out_features=32, bias=True)
  (fc3): Linear(in_features=32, out_features=1, bias=True)
)
